# Taller 2: Procesamiento y Persistencia de Datos con Spark

## Objetivo del Taller
Construir un pipeline de datos completo. El estudiante, como ingeniero de datos, debe:

1. Extraer datos de ventas desde un DataFrame creado manualmente
2. Almacenar en un Data Lake local (MinIO) en formato Delta Lake
3. Transformar con PySpark (limpieza, agregaciones, enriquecimiento)
4. Cargar resultados en PostgreSQL como Data Mart para reportes
5. Validar calidad de datos en cada etapa del pipeline


## 1. Celda 1: Configuración Inicial

In [2]:
# ============================================================
# PUNTO 1: PREPARACIÓN DEL AMBIENTE
# ============================================================

from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("TallerEvaluacionUnidad2")
    .master("local[*]")
    .config("spark.driver.memory", "1g")
    .config("spark.executor.memory", "1500m")

    # JDBC PostgreSQL
    .config("spark.jars", "/usr/local/spark-3.5.8/jars/postgresql-42.7.3.jar")

    # S3A MinIO
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    
    # Delta Lake
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate())

print("=" * 60)
print("VERIFICACIÓN DE SPARKSESSION")
print("=" * 60)

print(f"App Name: {spark.sparkContext.appName}")
print(f"Master: {spark.sparkContext.master}")
print(f"Spark Version: {spark.version}")
print(f"Python Version: {spark.sparkContext.pythonVer}")

26/06/16 15:36:34 WARN Utils: Your hostname, david-VMware-Virtual-Platform resolves to a loopback address: 127.0.1.1; using 172.16.179.130 instead (on interface ens33)
26/06/16 15:36:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/06/16 15:36:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


VERIFICACIÓN DE SPARKSESSION
App Name: TallerEvaluacionUnidad2
Master: local[*]
Spark Version: 3.5.8
Python Version: 3.12


## 2. Celda 2: Creación de Datos de Origen

In [4]:
# ============================================================
# PUNTO 2: CREACIÓN DE DATOS DE ORIGEN
# ============================================================
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType
from pyspark.sql.functions import col
from datetime import date
import os

schema_ventas = StructType([
    StructField("id_venta", StringType(), False),
    StructField("fecha", DateType(), False),
    StructField("id_producto", StringType(), False),
    StructField("producto", StringType(), False),
    StructField("categoria", StringType(), False),
    StructField("cantidad", IntegerType(), False),
    StructField("precio_unit", DoubleType(), False),
    StructField("id_cliente", StringType(), False),
    StructField("ciudad", StringType(), False),
    StructField("canal", StringType(), False),
])

data_ventas = [
    ("V001", date(2024, 1, 5),  "P001", "Laptop Pro",      "Electrónica",  2, 1200.00, "C001", "Bogotá",    "Online"),
    ("V002", date(2024, 1, 8),  "P002", "Mouse Inalámbrico","Accesorios",  5,   25.99, "C002", "Medellín",  "Tienda"),
    ("V003", date(2024, 1, 12), "P003", "Teclado Mecánico", "Accesorios",  3,   89.99, "C003", "Cali",      "Online"),
    ("V004", date(2024, 1, 15), "P001", "Laptop Pro",       "Electrónica", 1, 1200.00, "C004", "Barranquilla","Tienda"),
    ("V005", date(2024, 1, 18), "P004", "Monitor 4K",       "Electrónica", 2,  450.00, "C005", "Bogotá",    "Online"),
    ("V006", date(2024, 2, 3),  "P005", "Silla Ergonómica", "Muebles",     1,  320.00, "C001", "Medellín",  "Tienda"),
    ("V007", date(2024, 2, 10), "P002", "Mouse Inalámbrico","Accesorios",  8,   25.99, "C006", "Cali",      "Online"),
    ("V008", date(2024, 2, 14), "P006", "Webcam HD",        "Electrónica", 3,   75.00, "C007", "Bogotá",    "Tienda"),
    ("V009", date(2024, 2, 20), "P003", "Teclado Mecánico", "Accesorios",  2,   89.99, "C008", "Cartagena", "Online"),
    ("V010", date(2024, 3, 1),  "P005", "Silla Ergonómica", "Muebles",     2,  320.00, "C009", "Bogotá",    "Online"),
]

df_ventas_raw = spark.createDataFrame(data_ventas, schema=schema_ventas)


print("DataFrame creado exitosamente")
print(f"Total registros: {df_ventas_raw.count()}")
df_ventas_raw.printSchema()
df_ventas_raw.show(truncate=False)


DataFrame creado exitosamente


Total registros: 10
root
 |-- id_venta: string (nullable = false)
 |-- fecha: date (nullable = false)
 |-- id_producto: string (nullable = false)
 |-- producto: string (nullable = false)
 |-- categoria: string (nullable = false)
 |-- cantidad: integer (nullable = false)
 |-- precio_unit: double (nullable = false)
 |-- id_cliente: string (nullable = false)
 |-- ciudad: string (nullable = false)
 |-- canal: string (nullable = false)

+--------+----------+-----------+-----------------+-----------+--------+-----------+----------+------------+------+
|id_venta|fecha     |id_producto|producto         |categoria  |cantidad|precio_unit|id_cliente|ciudad      |canal |
+--------+----------+-----------+-----------------+-----------+--------+-----------+----------+------------+------+
|V001    |2024-01-05|P001       |Laptop Pro       |Electrónica|2       |1200.0     |C001      |Bogotá      |Online|
|V002    |2024-01-08|P002       |Mouse Inalámbrico|Accesorios |5       |25.99      |C002      |Medel

## 3. Celda 3: Actividad 2 - Transformaciones

In [6]:
# ============================================================
# PUNTO 3: DATA LAKE LOCAL - MINIO
# ============================================================
import boto3
from botocore.exceptions import ClientError
from pyspark.sql.functions import col, month, when, lit
from delta import DeltaTable

# 3.1 Verificar/crear bucket con boto3
print("=" * 60)
print("3.1 VERIFICACIÓN DE BUCKET MINIO")
print("=" * 60)

# Cliente S3 apuntando a MinIO
s3 = boto3.client(
    's3',
    endpoint_url='http://localhost:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin123',
    region_name='us-east-1'
)

bucket_name = "spark-lake"

try:
    s3.head_bucket(Bucket=bucket_name)
    print(f"\nBucket '{bucket_name}' ya existe")
except ClientError:
    s3.create_bucket(Bucket=bucket_name)
    print(f"\nBucket '{bucket_name}' creado")
    

# 3.2 Escribir datos crudos en Delta Lake sobre MinIO
print("\n" + "=" * 60)
print("3.2 ESCRITURA DELTA EN MINIO (ventas_raw)")
print("=" * 60)

delta_path_ventas = "s3a://spark-lake/datatech/ventas_raw"


print(f"\nEscribiendo en: {delta_path_ventas}")
df_ventas_raw.write.format("delta").mode("overwrite").save(delta_path_ventas)
print(f"✅ Datos crudos escritos en: {delta_path_ventas}")

# 3.3 Leer de vuelta y verificar integridad
df_raw_check = spark.read.format("delta").load(delta_path_ventas)

assert df_raw_check.count() == df_ventas_raw.count(), "❌ El conteo no coincide"
assert df_raw_check.columns == df_ventas_raw.columns, "❌ Las columnas no coinciden"
print(f"✅ Integridad verificada: {df_raw_check.count()} registros, columnas: {df_raw_check.columns}")


# 3.4 Transformaciones
print("\n" + "=" * 60)
print("3.4 TRANSFORMACIONES (ventas_online)")
print("=" * 60)

df_ventas_clean = (
    df_ventas_raw
    .withColumn("total_venta", col("cantidad") * col("precio_unit"))
    .withColumn("mes", month(col("fecha")))
    .withColumn("trimestre",
        when(col("mes").isin(1, 2, 3), lit("Q1"))
        .when(col("mes").isin(4, 5, 6), lit("Q2"))
        .when(col("mes").isin(7, 8, 9), lit("Q3"))
        .otherwise(lit("Q4"))
    )
    .withColumn("categoria_valor",
        when(col("total_venta") >= 1000, lit("Alto"))
        .when(col("total_venta") >= 200,  lit("Medio"))
        .otherwise(lit("Bajo"))
    )
    .withColumn("precio_con_iva", col("precio_unit") * lit(1.19))
    .filter(col("canal") == "Online")
)

print("DataFrame transformado (solo Online):")
df_ventas_clean.printSchema()
df_ventas_clean.show(truncate=False)


# 3.5 Escribir transformados en Delta
delta_online_path = "s3a://spark-lake/datatech/ventas_online"

df_ventas_clean.write.format("delta").mode("overwrite").save(delta_online_path)
print(f"\n✅ Datos transformados escritos en: {delta_online_path}")


# 3.6 Historial de versiones
print("\n" + "=" * 60)
print("3.6 HISTORIAL DE VERSIONES (ventas_online)")
print("=" * 60)

DeltaTable.forPath(spark, delta_online_path).history().select(
    "version", "timestamp", "operation", "operationParameters"
).show(truncate=False)

# 3.7 MERGE (UPSERT) en ventas_raw
print("\n" + "=" * 60)
print("3.7 MERGE (UPSERT) EN ventas_raw")
print("=" * 60)

nuevas_ventas = [
    # Registro existente — probará el UPDATE
    ("V001", date(2024, 3, 5),  "P001", "Laptop Pro",       "Electrónica", 3, 1250.00, "C001", "Bogotá",   "Online"),
    # Registros nuevos — probarán el INSERT
    ("V011", date(2024, 3, 10), "P006", "Webcam HD",         "Electrónica", 2,   75.00, "C010", "Medellín", "Tienda"),
    ("V012", date(2024, 3, 15), "P005", "Silla Ergonómica",  "Muebles",     1,  320.00, "C011", "Cali",     "Online"),
]

df_nuevas = spark.createDataFrame(nuevas_ventas, schema=schema_ventas)

(
    DeltaTable.forPath(spark, delta_path_ventas)
    .alias("target")
    .merge(
        df_nuevas.alias("source"),
        "target.id_venta = source.id_venta"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("✅ MERGE ejecutado exitosamente")

# 3.8 Historial actualizado
print("\n" + "=" * 60)
print("3.8 HISTORIAL ACTUALIZADO (ventas_raw)")
print("=" * 60)

DeltaTable.forPath(spark, delta_path_ventas).history().select(
    "version", "timestamp", "operation", "operationParameters"
).show(truncate=False)

df_post_merge = spark.read.format("delta").load(delta_path_ventas)
print(f"Total registros post-MERGE: {df_post_merge.count()}")
df_post_merge.orderBy("id_venta").show(truncate=False)

3.1 VERIFICACIÓN DE BUCKET MINIO

Bucket 'spark-lake' ya existe

3.2 ESCRITURA DELTA EN MINIO (ventas_raw)

Escribiendo en: s3a://spark-lake/datatech/ventas_raw


26/06/16 16:18:06 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

✅ Datos crudos escritos en: s3a://spark-lake/datatech/ventas_raw


26/06/16 16:18:14 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

✅ Integridad verificada: 10 registros, columnas: ['id_venta', 'fecha', 'id_producto', 'producto', 'categoria', 'cantidad', 'precio_unit', 'id_cliente', 'ciudad', 'canal']

3.4 TRANSFORMACIONES (ventas_online)
DataFrame transformado (solo Online):
root
 |-- id_venta: string (nullable = false)
 |-- fecha: date (nullable = false)
 |-- id_producto: string (nullable = false)
 |-- producto: string (nullable = false)
 |-- categoria: string (nullable = false)
 |-- cantidad: integer (nullable = false)
 |-- precio_unit: double (nullable = false)
 |-- id_cliente: string (nullable = false)
 |-- ciudad: string (nullable = false)
 |-- canal: string (nullable = false)
 |-- total_venta: double (nullable = false)
 |-- mes: integer (nullable = false)
 |-- trimestre: string (nullable = false)
 |-- categoria_valor: string (nullable = false)
 |-- precio_con_iva: double (nullable = false)

+--------+----------+-----------+-----------------+-----------+--------+-----------+----------+---------+------+-------

✅ MERGE ejecutado exitosamente

3.8 HISTORIAL ACTUALIZADO (ventas_raw)
+-------+-------------------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation|operationParameters                                                                                                                                                                      |
+-------+-------------------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|1      |2026-06-16 16:18:30|MERGE    |{predicate -> ["(id_venta#1808 = id_venta#1788)"], matchedPredicates -> [{"actionType":"update"}], notMatchedPredicates -> [{"actionType":"insert"}], notMatchedBySourcePredicates -> []}|
|0      |2026-06-16 16:18

Total registros post-MERGE: 12


+--------+----------+-----------+-----------------+-----------+--------+-----------+----------+------------+------+
|id_venta|fecha     |id_producto|producto         |categoria  |cantidad|precio_unit|id_cliente|ciudad      |canal |
+--------+----------+-----------+-----------------+-----------+--------+-----------+----------+------------+------+
|V001    |2024-03-05|P001       |Laptop Pro       |Electrónica|3       |1250.0     |C001      |Bogotá      |Online|
|V002    |2024-01-08|P002       |Mouse Inalámbrico|Accesorios |5       |25.99      |C002      |Medellín    |Tienda|
|V003    |2024-01-12|P003       |Teclado Mecánico |Accesorios |3       |89.99      |C003      |Cali        |Online|
|V004    |2024-01-15|P001       |Laptop Pro       |Electrónica|1       |1200.0     |C004      |Barranquilla|Tienda|
|V005    |2024-01-18|P004       |Monitor 4K       |Electrónica|2       |450.0      |C005      |Bogotá      |Online|
|V006    |2024-02-03|P005       |Silla Ergonómica |Muebles    |1       |

## 4. Celda 4: Actividad 3 - Consultas y Agregaciones

In [11]:
# ============================================================
# PUNTO 4: DATA MART RELACIONAL - POSTGRESQL
# ============================================================
from pyspark.sql.functions import sum as spark_sum, avg as spark_avg, count, max as spark_max
from pyspark.sql import Row
from datetime import date


# Configuración JDBC
jdbc_url = "jdbc:postgresql://localhost:5432/electiva"
connection_properties = {
"user": "postgres",
"password": "123abc",
"driver": "org.postgresql.Driver"
}

# 4.1 Leer Delta desde MinIO
print("=" * 60)
print("4.1 LECTURA DE DELTA DESDE MINIO")
print("=" * 60)

df_online = spark.read.format("delta").load(delta_online_path)
print(f"Registros leídos: {df_online.count()}")
df_online.show(5)

# 4.2 Crear vista temporal y consulta SQL
print("\n" + "=" * 60)
print("4.2 CONSULTA SQL - REPORTE TRIMESTRAL")
print("=" * 60)
df_online.createOrReplaceTempView("v_ventas_online")

df_reporte_sql = spark.sql("""
    SELECT
        categoria,
        trimestre,
        COUNT(*)                        AS num_ventas,
        SUM(total_venta)                AS total_ingresos,
        AVG(total_venta)                AS ticket_promedio,
        MAX(total_venta)                AS venta_maxima
    FROM v_ventas_online
    GROUP BY categoria, trimestre
    ORDER BY trimestre, total_ingresos DESC
""")

print("Resultado de la consulta SQL:")
df_reporte_sql.show()

# 4.3 Escribir reporte trimestral en PostgreSQL
print("\n" + "=" * 60)
print("4.3 ESCRITURA EN POSTGRESQL (reporte_trimestral)")
print("=" * 60)

df_reporte_sql.write.jdbc(
    url=jdbc_url,
    table="reporte_trimestral",
    mode="overwrite",
    properties=connection_properties
)
print("✅ Tabla 'reporte_trimestral' creada en PostgreSQL")

# 4.4 Crear tabla top_clientes_online
print("\n" + "=" * 60)
print("4.4 TABLA TOP_CLIENTES_ONLINE")
print("=" * 60)
df_top_clientes = (
    df_online
    .groupBy("id_cliente", "ciudad")
    .agg(
        spark_sum("total_venta").alias("total_gastado"),
        count("*").alias("num_compras"),
        spark_max("total_venta").alias("compra_maxima")
    )
    .filter(col("num_compras") > 1)
    .orderBy(col("total_gastado").desc())
)

print("Top clientes online (más de 1 compra):")
df_top_clientes.show()

df_top_clientes.write.jdbc(
    url=jdbc_url,
    table="top_clientes_online",
    mode="overwrite",
    properties=connection_properties
)
print("✅ Tabla 'top_clientes_online' creada en PostgreSQL")

# 4.5 Verificación de integridad
print("\n" + "=" * 60)
print("4.5 VERIFICACIÓN DE INTEGRIDAD")
print("=" * 60)

df_pg_reporte = spark.read.jdbc(
    url=jdbc_url,
    table="reporte_trimestral",
    properties=connection_properties
)

df_pg_clientes = spark.read.jdbc(
    url=jdbc_url,
    table="top_clientes_online",
    properties=connection_properties
)

reporte_spark = df_reporte_sql.count()
reporte_pg    = df_pg_reporte.count()
clientes_spark = df_top_clientes.count()
clientes_pg    = df_pg_clientes.count()

print(f"reporte_trimestral  — Spark: {reporte_spark}  | PostgreSQL: {reporte_pg}")
print(f"top_clientes_online — Spark: {clientes_spark} | PostgreSQL: {clientes_pg}")

assert reporte_spark == reporte_pg,    "❌ Conteo no coincide en reporte_trimestral"
assert clientes_spark == clientes_pg,  "❌ Conteo no coincide en top_clientes_online"
print("✅ Integridad verificada: conteos coinciden en ambas tablas")


# 4.6 Append incremental
print("\n" + "=" * 60)
print("4.6 APPEND INCREMENTAL")
print("=" * 60)


nuevo_registro = spark.createDataFrame([
    Row(
        categoria="Tecnología",
        trimestre="Q2",
        num_ventas=5,
        total_ingresos=3200.00,
        ticket_promedio=640.00,
        venta_maxima=1250.00
    )
])

nuevo_registro.write.jdbc(
    url=jdbc_url,
    table="reporte_trimestral",
    mode="append",
    properties=connection_properties
)

df_final = spark.read.jdbc(
    url=jdbc_url,
    table="reporte_trimestral",
    properties=connection_properties
)

print(f"✅ Registro Q2/Tecnología agregado. Total registros post-append: {df_final.count()}")
df_final.orderBy("trimestre", "categoria").show(truncate=False)


4.1 LECTURA DE DELTA DESDE MINIO


Registros leídos: 6
+--------+----------+-----------+-----------------+-----------+--------+-----------+----------+---------+------+------------------+---+---------+---------------+------------------+
|id_venta|     fecha|id_producto|         producto|  categoria|cantidad|precio_unit|id_cliente|   ciudad| canal|       total_venta|mes|trimestre|categoria_valor|    precio_con_iva|
+--------+----------+-----------+-----------------+-----------+--------+-----------+----------+---------+------+------------------+---+---------+---------------+------------------+
|    V007|2024-02-10|       P002|Mouse Inalámbrico| Accesorios|       8|      25.99|      C006|     Cali|Online|            207.92|  2|       Q1|          Medio|30.928099999999997|
|    V009|2024-02-20|       P003| Teclado Mecánico| Accesorios|       2|      89.99|      C008|Cartagena|Online|            179.98|  2|       Q1|           Bajo|107.08809999999998|
|    V010|2024-03-01|       P005| Silla Ergonómica|    Muebles|       2|   

✅ Registro Q2/Tecnología agregado. Total registros post-append: 4
+-----------+---------+----------+-----------------+------------------+------------------+
|categoria  |trimestre|num_ventas|total_ingresos   |ticket_promedio   |venta_maxima      |
+-----------+---------+----------+-----------------+------------------+------------------+
|Accesorios |Q1       |3         |657.8699999999999|219.28999999999996|269.96999999999997|
|Electrónica|Q1       |2         |3300.0           |1650.0            |2400.0            |
|Muebles    |Q1       |1         |640.0            |640.0             |640.0             |
|Tecnología |Q2       |5         |3200.0           |640.0             |1250.0            |
+-----------+---------+----------+-----------------+------------------+------------------+



## 5. Celda 5: Actividad 4 - Modificación y Persistencia

In [14]:
# ============================================================
# PUNTO 5: PIPELINE END-TO-END Y VALIDACIÓN DE CALIDAD
# ============================================================
from pyspark.sql.functions import col, isnan, when, count, abs as spark_abs, lit
from pyspark.sql.types import TimestampType, StructType, StructField, StringType, IntegerType
from pyspark.sql import Row
from datetime import datetime

# 5.1 Función de validación de calidad
def validar_calidad(df, nombre_tabla):
    """
    lida la calidad de datos de un DataFrame.
    Retorna (True/False, count) si pasa/falla todas las validaciones.
    """
    print(f"\n{'='*60}")
    print(f"VALIDACIÓN DE CALIDAD: {nombre_tabla}")
    print(f"{'='*60}")
    
    total = df.count()
    resultados = []

    # Validación 1: Completitud — detectar nulos en columnas clave
    columnas_clave = ["id_venta", "fecha", "id_producto", "cantidad", "precio_unit"]
    nulos = df.select([
        count(when(col(c).isNull() | isnan(col(c)), c)).alias(c)
        for c in columnas_clave
        if dict(df.dtypes)[c] in ("double", "float", "int", "bigint", "string")
    ])
    total_nulos = sum(nulos.first())
    pass1 = total_nulos == 0
    resultados.append(("Completitud (sin nulos en columnas clave)", pass1, f"{total_nulos} nulos encontrados"))

    # Validación 2: Unicidad — detectar id_venta duplicados
    if "id_venta" in df.columns:
        duplicados = df.groupBy("id_venta").count().filter(col("count") > 1).count()
        pass2 = duplicados == 0
        resultados.append(("Unicidad (id_venta sin duplicados)", pass2, f"{duplicados} duplicados encontrados"))
    else:
        pass2 = True
        resultados.append(("Unicidad (id_venta)", True, "Columna no aplica en esta tabla"))

    # Validación 3: Rango — cantidad debe ser mayor a 0
    if "cantidad" in df.columns:
        fuera_rango = df.filter(col("cantidad") <= 0).count()
        pass3 = fuera_rango == 0
        resultados.append(("Rango (cantidad > 0)", pass3, f"{fuera_rango} registros con cantidad <= 0"))
    else:
        pass3 = True
        resultados.append(("Rango (cantidad)", True, "Columna no aplica en esta tabla"))

    # Validación 4: Consistencia — total_venta == cantidad * precio_unit
    if "total_venta" in df.columns and "cantidad" in df.columns and "precio_unit" in df.columns:
        inconsistentes = df.filter(
            spark_abs(col("total_venta") - (col("cantidad") * col("precio_unit"))) > 0.01
        ).count()
        pass4 = inconsistentes == 0
        resultados.append(("Consistencia (total_venta == cantidad * precio_unit)", pass4, f"{inconsistentes} registros inconsistentes"))
    else:
        pass4 = True
        resultados.append(("Consistencia (total_venta)", True, "Columnas no aplican en esta tabla"))

    # Imprimir reporte
    print(f"{'Validación':<50} {'Estado':<8} Detalle")
    print("-" * 80)
    for nombre, paso, detalle in resultados:
        estado = "✅ PASS" if paso else "❌ FAIL"
        print(f"{nombre:<50} {estado:<8} {detalle}")

    print(f"\nTotal registros: {total}")
    all_pass = all(r[1] for r in resultados)
    print(f"Resultado general: {'✅ PASS' if all_pass else '❌ FAIL'}")

    return all_pass, total


# 5.2 Ejecutar validaciones en cada etapa
print("=" * 60)
print("5.2 VALIDACIONES EN CADA ETAPA DEL PIPELINE")
print("=" * 60)

log_pipeline = []

# Etapa 1 — Validar df_ventas_raw (datos originales en memoria)
ok1, count1 = validar_calidad(df_ventas_raw, "ventas_raw (origen)")
log_pipeline.append(("ventas_raw_origen", count1, "PASS" if ok1 else "FAIL"))

# Etapa 2 — Validar ventas_raw post-MERGE desde MinIO
df_ventas_raw_minio = spark.read.format("delta").load(delta_path_ventas)
ok2, count2 = validar_calidad(df_ventas_raw_minio, "ventas_raw post-MERGE (MinIO)")
log_pipeline.append(("ventas_raw_post_merge", count2, "PASS" if ok2 else "FAIL"))

# Etapa 3 — Validar ventas_online desde MinIO
df_ventas_online_minio = spark.read.format("delta").load(delta_online_path)
ok3, count3 = validar_calidad(df_ventas_online_minio, "ventas_online (MinIO)")
log_pipeline.append(("ventas_online", count3, "PASS" if ok3 else "FAIL"))

# Etapa 4 — Validar reporte_trimestral desde PostgreSQL
# id_venta no existe en esta tabla, solo se verifica el conteo
df_pg_reporte = spark.read.jdbc(
    url=jdbc_url,
    table="reporte_trimestral",
    properties=connection_properties
)
count4 = df_pg_reporte.count()
print(f"\n{'='*60}")
print("VALIDACIÓN DE CALIDAD: reporte_trimestral (PostgreSQL)")
print(f"{'='*60}")
print(f"Total registros: {count4}")
print("✅ PASS — tabla leída correctamente desde PostgreSQL")
log_pipeline.append(("reporte_trimestral_pg", count4, "PASS"))


# 5.3 Reporte final del pipeline
print("\n" + "=" * 60)
print("5.3 REPORTE FINAL DEL PIPELINE")
print("=" * 60)

data_for_reporte = []
for etapa, registros, validacion in log_pipeline:
    data_for_reporte.append((etapa, registros, validacion, datetime.now()))

schema_reporte = StructType([
    StructField("etapa",      StringType(),   False),
    StructField("registros",  IntegerType(),  False),
    StructField("validacion", StringType(),   False),
    StructField("timestamp",  TimestampType(), False),
])

df_reporte_pipeline = spark.createDataFrame(data_for_reporte, schema=schema_reporte)

print("Reporte del pipeline:")
df_reporte_pipeline.show(truncate=False)

df_reporte_pipeline.write.jdbc(
    url=jdbc_url,
    table="log_pipeline",
    mode="overwrite",
    properties=connection_properties
)
print("\n✅ Tabla 'log_pipeline' creada en PostgreSQL")

# Verificación final
df_log_pg = spark.read.jdbc(
    url=jdbc_url,
    table="log_pipeline",
    properties=connection_properties
)
print(f"Registros en log_pipeline (PostgreSQL): {df_log_pg.count()}")
df_log_pg.show(truncate=False)


5.2 VALIDACIONES EN CADA ETAPA DEL PIPELINE

VALIDACIÓN DE CALIDAD: ventas_raw (origen)
Validación                                         Estado   Detalle
--------------------------------------------------------------------------------
Completitud (sin nulos en columnas clave)          ✅ PASS   0 nulos encontrados
Unicidad (id_venta sin duplicados)                 ✅ PASS   0 duplicados encontrados
Rango (cantidad > 0)                               ✅ PASS   0 registros con cantidad <= 0
Consistencia (total_venta)                         ✅ PASS   Columnas no aplican en esta tabla

Total registros: 10
Resultado general: ✅ PASS

VALIDACIÓN DE CALIDAD: ventas_raw post-MERGE (MinIO)


Validación                                         Estado   Detalle
--------------------------------------------------------------------------------
Completitud (sin nulos en columnas clave)          ✅ PASS   0 nulos encontrados
Unicidad (id_venta sin duplicados)                 ✅ PASS   0 duplicados encontrados
Rango (cantidad > 0)                               ✅ PASS   0 registros con cantidad <= 0
Consistencia (total_venta)                         ✅ PASS   Columnas no aplican en esta tabla

Total registros: 12
Resultado general: ✅ PASS

VALIDACIÓN DE CALIDAD: ventas_online (MinIO)


Validación                                         Estado   Detalle
--------------------------------------------------------------------------------
Completitud (sin nulos en columnas clave)          ✅ PASS   0 nulos encontrados
Unicidad (id_venta sin duplicados)                 ✅ PASS   0 duplicados encontrados
Rango (cantidad > 0)                               ✅ PASS   0 registros con cantidad <= 0
Consistencia (total_venta == cantidad * precio_unit) ✅ PASS   0 registros inconsistentes

Total registros: 6
Resultado general: ✅ PASS

VALIDACIÓN DE CALIDAD: reporte_trimestral (PostgreSQL)
Total registros: 4
✅ PASS — tabla leída correctamente desde PostgreSQL

5.3 REPORTE FINAL DEL PIPELINE
Reporte del pipeline:
+---------------------+---------+----------+--------------------------+
|etapa                |registros|validacion|timestamp                 |
+---------------------+---------+----------+--------------------------+
|ventas_raw_origen    |10       |PASS      |2026-06-16 16:46:41.